# 🖐️ Touchless HCI: Real-Time Hand Tracking
**Project Type:** YOLO Object Detection & Local Edge Inference

**Team:** Ahmed Khalid, Omar A. El Nasser, Bassel Adel, Karim Hossam, Mohamed Haitham, Omar Bekhiet

---

## 📖 Quick Setup & Execution Guide
Welcome to the training and inference notebook for the Touchless HCI Hand Tracker.

### ☁️ Option A: Google Colab (Cloud GPU Training)
1. **Upload Models:** If you already have `best.onnx` generated, upload it to your Colab session storage.
2. **Install Dependencies:** Run the first setup cells to install `ultralytics` and `roboflow`.
3. **Table of Contents:** Navigate directly to the section you wish to execute.
4. *⚠️ Note: Live webcam inference is **not supported** natively in Google Colab. Use static image testing instead.*

### 💻 Option B: Local Execution (VSCode / PyCharm)
1. Ensure the `best.onnx` model is located in the exact same directory as this Jupyter Notebook.
2. Navigate to the **"Load Model in ONNX Format"** inference section and execute.
3. *✅ Note: You can successfully run the live webcam inference cells locally!*

In [4]:
!pip install ultralytics roboflow

  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)


In [ ]:
%pip install opencv-python

---
## 🔐 1. Initialize Roboflow Workspace
*(Authenticates the API key and connects to the `hand-detection-project` workspace to prepare for dataset imports)*

In [6]:
from roboflow import Roboflow

rf = Roboflow(api_key="bBVUlQtfrSkIzoma52N9")

project = rf.workspace("ahmed-kamel").project("hand-detection-project")

loading Roboflow workspace...
loading Roboflow project...


## 📦 2. Download Version 1 Dataset
*(Fetches the initial V1 dataset structure and annotations formatted specifically for YOLO architecture)*

In [ ]:
from roboflow import Roboflow
print("--- DOWNLOADING VERSION 1 ---")
dataset_v1 = project.version(1).download("yolov8")

## 🚀 2. Train Model Version 1
*(Trains the YOLO architecture on the V1 dataset for 25 epochs)*

In [ ]:
from ultralytics import YOLO
print("--- TRAINING VERSION 1 ---")
model_v1 = YOLO('yolo26n.pt')
results_v1 = model_v1.train(data=f"{dataset_v1.location}/data.yaml", epochs=25, imgsz=512, name='v1_results')

---
## 📦 3. Download Model Version 2 Dataset
*(Fetches the updated V2 dataset)*

In [ ]:
from roboflow import Roboflow
print("--- DOWNLOADING VERSION 2 ---")
dataset_v2 = project.version(2).download("yolov8")

## 🚀 4. Train Model Version 2
*(Trains the YOLO architecture on the V2 dataset for 25 epochs)*

In [ ]:
from ultralytics import YOLO
print("--- TRAINING VERSION 2 ---")
model_v2 = YOLO('yolo26n.pt')

results_v2 = model_v2.train(data=f"{dataset_v2.location}/data.yaml",
                            epochs=25,
                            imgsz=512,
                            name='v2_results')

## 💾 5. Save Version 2 Weights to Google Drive
*(Backs up the `best.pt` file to prevent data loss when the Colab instance resets)*

In [ ]:
from google.colab import drive
import shutil

print("Mounting Google Drive...")
drive.mount('/content/drive')

source_file = '/content/runs/detect/v2_results/weights/best.pt'
drive_destination = '/content/drive/MyDrive/best_v2.pt'

print("Copying file...")
shutil.copy(source_file, drive_destination)

print("Success! The file is now in your Google Drive.")

---
## 📦 6. Download Model Version 3 Dataset (Final 12K Images)
*(Downloads the 12,000 image dataset for the final robust model)*

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="bBVUlQtfrSkIzoma52N9")

project = rf.workspace("ahmed-kamel").project("hand-detection-project")

dataset_v3 = project.version(3).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Hand-Detection-Project-3 in yolov8:: 100%|██████████| 25289/25289 [00:03<00:00, 7333.51it/s]


## 🛠️ 7. Dataset Engineering: 73-to-1 Class Consolidation
*(A critical preprocessing step that forcefully iterates through thousands of `.txt` labels to overwrite 73 distinct ASL gesture classes into a unified `0: hand` class. Updates `data.yaml` and clears old memory caches.)*

In [ ]:
import os
import yaml
import glob

print("Starting dataset overwrite...")

yaml_path = '/content/Hand-Detection-Project-3/data.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['nc'] = 1
data['names'] = ['hand']

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
print("1. data.yaml permanently updated to nc=1")

def convert_labels(label_dir):
    txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                parts[0] = '0'
                new_lines.append(' '.join(parts) + '\n')

        with open(txt_file, 'w') as f:
            f.writelines(new_lines)

for split in ['train', 'valid', 'test']:
    label_dir = f'/content/Hand-Detection-Project-3/{split}/labels'
    cache_file = f'/content/Hand-Detection-Project-3/{split}/labels.cache'

    if os.path.exists(label_dir):
        convert_labels(label_dir)
        print(f"2. Converted all text files in the {split} folder.")

    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"3. Deleted old {split} memory cache.")

print("Success! All 12,000 images are now unified into a single 'hand' class.")

Starting dataset overwrite...
1. data.yaml permanently updated to nc=1
2. Converted all text files in the train folder.
2. Converted all text files in the valid folder.
2. Converted all text files in the test folder.
Success! All 12,000 images are now unified into a single 'hand' class.


## 💾 8. Mount Drive for Version 3 Output
*(Sets up the Google Drive pipeline to automatically save checkpoints during the heavy V3 training)*

In [ ]:
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/yolo_runs'
print(f"Weights will be saved to: {DRIVE_PROJECT}/v3_results/weights/")

Mounting Google Drive...
Mounted at /content/drive
Weights will be saved to: /content/drive/MyDrive/yolo_runs/v3_results/weights/


## 🚀 9. Train Final Model (Version 3)
*(Initiates the 100-epoch training cycle using an NVIDIA Tesla T4 GPU. Implements an early stopping mechanism `patience=15` to prevent overfitting.)*

In [ ]:
from ultralytics import YOLO
model_v3 = YOLO('yolo26n.pt')
results_v3 = model_v3.train(
    data=f"{dataset_v3.location}/data.yaml",
    epochs=100,
    patience=15,
    batch=32,
    imgsz=512,
    project=DRIVE_PROJECT,
    name='v3_results',
    save_period=10
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Hand-Detection-Project-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v3_results, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

## ♻️ 10. Resume Training (Optional Recovery)
*(Execute this cell **only** if the previous training run was interrupted or disconnected. It will seamlessly resume from the last saved `epoch.pt` checkpoint.)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO

model_v3 = YOLO('/content/drive/MyDrive/yolo_runs/v3_results/weights/epoch90.pt')
model_v3.train(resume=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Hand-Detection-Project-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDri

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b38e2e468a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

---
## ⚙️ 11. Export & Load Model in ONNX Format
*(Converts the heavy PyTorch `.pt` model into an optimized ONNX graph for high-speed edge hardware inference)*

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import shutil

print("Mounting Google Drive...")
drive.mount('/content/drive')

model = YOLO('/content/drive/MyDrive/yolo_runs/v3_results/weights/best.pt')
model.export(format='onnx')
print("ONNX saved to Drive!")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/yolo_runs/v3_results/weights/best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: export success ✅ 1.8s, saved as '/content/drive/MyDrive/yolo_runs/v3_results/weights/best.onnx' (9.3 MB)

Export complete (2.1s)
Results saved to /content/drive/.shortcut-targets-by-id/19KiHat8kobWmMIwnxhm0VoskFH4kCtg4/yolo_runs/v3_results/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/yolo_runs/v3_results/weights/best.onnx imgsz=512 
Validate:        yolo val task=detect model=/content/drive/MyDrive/yolo_runs/v3_results/weights/best.onnx imgsz=512 data=/content/Hand-Detection-Project-3/data.yaml  
Visualize:       https://netron.app
ONNX saved to Drive!


# **Save The Results Of Model Verison 3 In The Drive**

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

folder_to_save = '/content/drive/MyDrive/yolo_runs/v3_results'
zip_filename = '/content/v3_results_complete'

print("Zipping...")
shutil.make_archive(zip_filename, 'zip', folder_to_save)

print("Copying to Drive...")
shutil.copy('/content/v3_results_complete.zip', '/content/drive/MyDrive/v3_results_complete.zip')

print("Done! ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zipping...
Copying to Drive...
Done! ✅


## 📸 12. Inference: Test on Static Images
*(Runs the compiled model against unseen test images to verify the Intersection over Union accuracy)*

In [3]:
from ultralytics import YOLO
model = YOLO('best.onnx')
result = model.predict("https://upload.wikimedia.org/wikipedia/commons/thumb/9/92/2025.04.26_%E3%80%8C%E5%8F%8D%E7%B6%A0%E5%85%B1%EF%BC%8E%E6%88%B0%E7%8D%A8%E8%A3%81%E3%80%8D%E5%87%B1%E9%81%93%E9%9B%86%E7%B5%90_099_%28cropped%29.jpg/960px-2025.04.26_%E3%80%8C%E5%8F%8D%E7%B6%A0%E5%85%B1%EF%BC%8E%E6%88%B0%E7%8D%A8%E8%A3%81%E3%80%8D%E5%87%B1%E9%81%93%E9%9B%86%E7%B5%90_099_%28cropped%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail")
for result in result:
    result.show()

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider

WARNING Download failure, retrying 1/3 https://upload.wikimedia.org/wikipedia/commons/thumb/9/92/2025.04.26__099_(cropped).jpg/960px-2025.04.26__099_(cropped).jpg... HTTP Error 403: Forbidden
image 1/1 C:\Users\ahmed\Documents\Neural Networks\HandDetectionProject\960px-2025.04.26__099_(cropped).jpg: 512x512 1 hand, 108.7ms
Speed: 52.4ms preprocess, 108.7ms inference, 40.6ms postprocess per image at shape (1, 3, 512, 512)


## 🎥 13. Inference: Live Webcam Tracking
*(Initializes OpenCV to capture live video frames and draw bounding boxes in real-time. **Must be run locally in VSCode/PyCharm, not Colab**)*

In [1]:
from ultralytics import YOLO
import cv2

model = YOLO('best.onnx')
cap = cv2.VideoCapture(0)

print("Starting webcam... Press 'q' to exit.")

while cap.isOpened():

    success, frame = cap.read()
    if not success:
        print("Failed to grab frame.")
        break

    results = model(frame, conf=0.5, verbose=False)

    annotated_frame = results[0].plot()

    cv2.imshow('Touchless HCI - Live Hand Detection', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Starting webcam... Press 'q' to exit.
Loading best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider
